<a href="https://colab.research.google.com/github/SuyaNangabo/plp-da-a2-rainfall-advisory/blob/main/rainfall_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Rainfall Analysis and Planting Advisory

Monthly rainfall readings (mm, January to December) for five counties are cleaned, combined into a NumPy array and analysed to recommend planting windows for SMS advice to farmers.

## 1. Setup and Data

In [46]:
import csv
import numpy as np

counties = ["Nairobi", "Nakuru", "Kisumu", "Mombasa", "Machakos"]

months = np.array(["January", "February", "March", "April", "May", "June",
                   "July", "August", "September", "October", "November", "December"])

clean_rows = [
    [64, 56, 92, 219, 176, 35, 18, 24, 31, 61, 165, 104],   # Nairobi
    [35, 42, 70, 145, 120, 78, 96, 105, 72, 55, 60, 38],    # Nakuru
    [58, 90, 150, 240, 180, 95, 70, 85, 90, 88, 130, 95],   # Kisumu
    [30, 15, 55, 150, 240, 110, 90, 65, 60, 90, 100, 70],   # Mombasa
]

machakos_raw = ["48", "35", "88", "N/A", "140", " 20 ",
                "9", "12", "", "45", "182", "96"]

## 2. Cleaning the Machakos Data

Each raw value is converted with `int()`, which also strips surrounding spaces such as `" 20 "`. Values that cannot be converted raise a `ValueError` and are replaced with the rounded average of the valid Machakos values.

In [47]:
def clean_rainfall(raw_values, month_names):
    parsed = []
    for value in raw_values:
        try:
            parsed.append(int(value))
        except ValueError:
            parsed.append(None)

    valid = [v for v in parsed if v is not None]
    fill_value = round(sum(valid) / len(valid))

    cleaned = [fill_value if v is None else v for v in parsed]
    imputed_months = [month_names[i] for i, v in enumerate(parsed) if v is None]
    return cleaned, imputed_months, fill_value


machakos_clean, imputed_months, fill_value = clean_rainfall(machakos_raw, months)

print("Raw values:     ", machakos_raw)
print("Cleaned values: ", machakos_clean)
print(f"Average of valid values used for imputation: {fill_value} mm")
for m in imputed_months:
    print(f"Imputed month: {m} -> {fill_value} mm")

Raw values:      ['48', '35', '88', 'N/A', '140', ' 20 ', '9', '12', '', '45', '182', '96']
Cleaned values:  [48, 35, 88, 68, 140, 20, 9, 12, 68, 45, 182, 96]
Average of valid values used for imputation: 68 mm
Imputed month: April -> 68 mm
Imputed month: September -> 68 mm


## 3. Combined 5 × 12 Array

In [48]:
R = np.array(clean_rows + [machakos_clean])

print("Shape:", R.shape)
print("Dtype:", R.dtype)
print(R)

Shape: (5, 12)
Dtype: int64
[[ 64  56  92 219 176  35  18  24  31  61 165 104]
 [ 35  42  70 145 120  78  96 105  72  55  60  38]
 [ 58  90 150 240 180  95  70  85  90  88 130  95]
 [ 30  15  55 150 240 110  90  65  60  90 100  70]
 [ 48  35  88  68 140  20   9  12  68  45 182  96]]


## 4. Annual Totals and Monthly Averages

In [49]:
annual_totals = R.sum(axis=1)
monthly_averages = R.mean(axis=0)

print("Annual total rainfall per county (mm)")
for county, total in zip(counties, annual_totals):
    print(f"{county:<10} {total}")

print()
print("Average rainfall per month across all counties (mm)")
for month, avg in zip(months, monthly_averages):
    print(f"{month:<10} {avg:.1f}")

Annual total rainfall per county (mm)
Nairobi    1045
Nakuru     916
Kisumu     1371
Mombasa    1075
Machakos   811

Average rainfall per month across all counties (mm)
January    47.0
February   47.6
March      91.0
April      164.4
May        171.2
June       67.6
July       56.6
August     58.2
September  64.2
October    67.8
November   127.4
December   80.6


## 5. Wettest Month per County

In [50]:
wettest_idx = np.argmax(R, axis=1)
wettest_months = months[wettest_idx]
wettest_values = R[np.arange(R.shape[0]), wettest_idx]

for county, month, value in zip(counties, wettest_months, wettest_values):
    print(f"{county:<10} {month} ({value} mm)")

Nairobi    April (219 mm)
Nakuru     April (145 mm)
Kisumu     April (240 mm)
Mombasa    May (240 mm)
Machakos   November (182 mm)


## 6. Dry Months (below 50 mm)

In [51]:
dry_mask = R < 50
dry_counts = dry_mask.sum(axis=1)

for county, count in zip(counties, dry_counts):
    print(f"{county:<10} {count} dry months")

Nairobi    4 dry months
Nakuru     3 dry months
Kisumu     0 dry months
Mombasa    2 dry months
Machakos   6 dry months


## 7. Planting Window

Adding each month to the following month with `R[:, :-1] + R[:, 1:]` gives a 5 × 11 array of two-month totals. The highest pair per county is the recommended planting window.

In [52]:
pair_totals = R[:, :-1] + R[:, 1:]
start_idx = np.argmax(pair_totals, axis=1)
window_totals = pair_totals[np.arange(R.shape[0]), start_idx]

planting_windows = np.char.add(np.char.add(months[start_idx], "-"), months[start_idx + 1])

print("Pair totals shape:", pair_totals.shape)
for county, window, total in zip(counties, planting_windows, window_totals):
    print(f"{county:<10} {window} ({total} mm)")

Pair totals shape: (5, 11)
Nairobi    April-May (395 mm)
Nakuru     April-May (265 mm)
Kisumu     April-May (420 mm)
Mombasa    April-May (390 mm)
Machakos   November-December (278 mm)


## 8. Writing rainfall_summary.csv

In [53]:
with open("rainfall_summary.csv", "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["county", "annual_total_mm", "wettest_month", "dry_months", "planting_window"])
    for row in zip(counties, annual_totals, wettest_months, dry_counts, planting_windows):
        writer.writerow([row[0], int(row[1]), str(row[2]), int(row[3]), str(row[4])])

with open("rainfall_summary.csv", newline="") as f:
    rows = list(csv.reader(f))

for row in rows:
    print(row)
print("Data rows:", len(rows) - 1)

['county', 'annual_total_mm', 'wettest_month', 'dry_months', 'planting_window']
['Nairobi', '1045', 'April', '4', 'April-May']
['Nakuru', '916', 'April', '3', 'April-May']
['Kisumu', '1371', 'April', '0', 'April-May']
['Mombasa', '1075', 'May', '2', 'April-May']
['Machakos', '811', 'November', '6', 'November-December']
Data rows: 5


## Planting Advisory

Nairobi, Nakuru, Kisumu and Mombasa all peak in April and May, so farmers in these counties should prepare land in March and plant at the onset of the long rains in late March or early April. Kisumu is the wettest county at 1371 mm a year with no dry months, which makes it the most reliable for a long rains crop and a second crop during the October to December short rains. Machakos is the driest county at 811 mm with six dry months, so farmers there should plant drought tolerant crops such as sorghum, green grams or early maturing maize with the short rains in late October to use the November to December peak. Nairobi has four dry months from June to September, so long rains crops there should be early maturing varieties that are harvested before the dry spell. Machakos April and September readings were missing and were filled with an average value, so Machakos advice should be confirmed against station records before the SMS goes out.